# Week 2 - Data Pipeline and Training Preparation

Goals: train/val/test splits, PyTorch Dataset, Albumentations, DataLoaders, and a first binary U-Net baseline.

In [ ]:
from pathlib import Path
import importlib
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT / 'src'))

import week2_dataset
import week2_model
importlib.reload(week2_dataset)
importlib.reload(week2_model)

from week2_dataset import create_train_val_test_splits, build_dataloaders
from week2_model import UNet

DATA_DIR = PROJECT_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'splits'

## 1. Create Train / Val / Test Splits

In [ ]:
splits = create_train_val_test_splits(DATA_DIR, SPLIT_DIR, seed=42)
{name: len(ids) for name, ids in splits.items()}

## 2. Build DataLoaders

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(
    data_dir=DATA_DIR,
    split_dir=SPLIT_DIR,
    image_size=512,
    batch_size=2,
    num_workers=0,
    target_mode='binary',
)

batch = next(iter(train_loader))
batch['image'].shape, batch['mask'].shape, batch['sample_id'][:2]

Expected shapes:

- image: `batch x 6 x 512 x 512`
- mask: `batch x 1 x 512 x 512`

## 3. Check Baseline U-Net Forward Pass

In [ ]:
model = UNet(in_channels=6, out_channels=1)
logits = model(batch['image'])
logits.shape

## 4. Train From Terminal

```bash
python src/week2_dataset.py
python src/week2_train_baseline.py --epochs 5 --batch-size 4 --image-size 512
```

The first baseline predicts binary building masks from stacked pre/post images.